# **WEEK 2 — Box–Jenkins Pipeline (ARMA/ARIMA)**

>Jerald B. Bongalos | PhD in Data Science | Asian Institute of Management

---

*Reference:*
>Shumway, R. H., & Stoffer, D. S. *Time Series Analysis and Its Applications*
>(Identification → Estimation → Diagnostics → Iteration)

---

**Purpose of this Notebook**

This notebook develops the Box–Jenkins methodology as a **statistical pipeline**, not as an automated modeling routine.

**The goal is to understand:**

- what information ACF and PACF can and cannot provide,
- why causality and invertibility are distinct and necessary,
- how differencing fundamentally alters dependence structure,
- why diagnostics override information criteria,
- and how seasonal structure requires explicit modeling (SARIMA).

**Learning Outcomes**

By the end of this notebook, you should be able to:

1. Identify plausible ARIMA models using ACF/PACF with confidence limits
2. Explain causality vs invertibility in AR and MA models
3. Explain what differencing does to ACF and the spectrum
4. Fit ARIMA models with justification (not automation)
5. Diagnose model adequacy using residual analysis
6. Explain why "lowest AIC" can still be wrong
7. Recognize when seasonal ARIMA (SARIMA) is required
8. Defend a final model selection in an exam setting

---

**Notebook Structure**

| Part | Topic | Type |
|------|-------|------|
| 1 | Box–Jenkins Philosophy | Theory |
| 2 | Data-Generating Process (Simulated PM2.5) | Theory + Simulation |
| 3 | Identification Using ACF/PACF | Theory + Simulation |
| 4 | Causality vs Invertibility | Theory + Simulation |
| 5 | Differencing: ACF and Spectrum Effects | Theory + Simulation |
| 6 | Candidate ARIMA Fitting | Theory + Simulation |
| 7 | Diagnostics: Where Models Live or Die | Theory + Simulation |
| 8 | Why "Lowest AIC" Can Still Be Wrong | Theory + Simulation |
| 9 | SARIMA: Handling Seasonality Properly | Theory + Simulation |
| 10 | Final Model Defense (Exam Template) | Synthesis |
| 11 | Self-Test Questions | Exam Preparation |


In [ ]:
# ============================================================
# SETUP
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import acf, pacf, adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy.stats import norm

np.random.seed(123)

# --- Shared periodogram helper ---
def periodogram_plot(x, title, fs=1.0, max_freq=0.5):
    """Plot raw periodogram."""
    x = np.asarray(x) - np.mean(x)
    n = len(x)
    freqs = np.fft.rfftfreq(n, d=1/fs)
    Pxx = (np.abs(np.fft.rfft(x))**2) / n
    mask = freqs <= max_freq
    plt.figure(figsize=(12, 3))
    plt.plot(freqs[mask], Pxx[mask], linewidth=1.2, color="#2c3e50")
    plt.title(title, fontsize=12)
    plt.xlabel("Frequency (cycles/month)")
    plt.ylabel("Power")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# --- Shared residual diagnostics ---
def residual_diagnostics(res, name, lags=(12, 24, 36), alpha=0.05):
    """Standard residual diagnostic suite for ARIMA models."""
    r = res.resid.dropna()

    fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))

    # Residual time plot
    axes[0].plot(r.index, r.values, linewidth=0.8, color="#2c3e50")
    axes[0].axhline(0, color="gray", linestyle="--", alpha=0.5)
    axes[0].set_title(f"{name} — Residuals", fontsize=11)
    axes[0].grid(True, alpha=0.3)

    # Residual ACF
    a = acf(r, nlags=48, fft=True)
    ci = 1.96 / np.sqrt(len(r))
    axes[1].stem(range(len(a)), a, basefmt=" ", markerfmt=".", linefmt="-")
    axes[1].axhline(ci, color="red", linestyle="--", alpha=0.5)
    axes[1].axhline(-ci, color="red", linestyle="--", alpha=0.5)
    axes[1].set_title(f"{name} — Residual ACF", fontsize=11)
    axes[1].set_xlabel("Lag")
    axes[1].grid(True, alpha=0.3)

    # Residual histogram + Normal overlay
    axes[2].hist(r, bins=30, density=True, alpha=0.7, color="#3498db", edgecolor="white")
    xg = np.linspace(r.min(), r.max(), 200)
    axes[2].plot(xg, norm.pdf(xg, r.mean(), r.std()), "r-", linewidth=2, label="Normal")
    axes[2].set_title(f"{name} — Residual Distribution", fontsize=11)
    axes[2].legend(fontsize=8)
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Ljung-Box
    lb = acorr_ljungbox(r, lags=list(lags), return_df=True)
    print(f"\n=== Ljung-Box for {name} (alpha={alpha}) ===")
    print(lb.to_string(index=True))

    if (lb["lb_pvalue"] < alpha).any():
        print(f"  Result: NOT white (autocorrelation remains).")
    else:
        print(f"  Result: Approximately white (no significant autocorrelation).")

print("Setup complete.")


## **PART 1 — Box–Jenkins Philosophy**

> **Why this part matters**
>
> Box–Jenkins is not "try models until one fits." It is a **disciplined iterative pipeline**:
> Identification → Estimation → Diagnostics → Iteration.
>
> The series must be **fixed before modeling begins**. Changing the series mid-pipeline
> invalidates model comparison.

---

### **1.1 The Pipeline**

1. **Transform** the series to stabilize variance (log, Box–Cox)
2. **Difference** to remove trend and/or seasonality → achieve stationarity
3. **Identify** candidate models from ACF/PACF patterns
4. **Estimate** parameters (MLE or conditional sum of squares)
5. **Diagnose** residuals (whiteness, normality, stability)
6. **Iterate** — reject inadequate models, expand candidates

---

### **1.2 What ACF/PACF Can and Cannot Do**

In theory:
- AR($p$) → PACF cuts off after lag $p$
- MA($q$) → ACF cuts off after lag $q$
- ARMA($p,q$) → both tail off

In practice:
- These rules are **asymptotic heuristics**, not identification guarantees
- Finite samples blur cutoffs
- Near-unit-root behavior distorts patterns
- Multiple models can match the same ACF/PACF

> **Exam-safe statement:** "ACF and PACF guide identification, but they do not determine the model. They suggest candidates; diagnostics decide."

---

### **1.3 Modeling Target**

In this notebook, we model $\log(\text{PM2.5})$, following variance stabilization.
All identification and diagnostics refer to this transformed series unless stated otherwise.


## **PART 2 — Data-Generating Process: Simulated Monthly PM2.5**

> **Why simulated data?**
>
> Box–Jenkins is about identifying and modeling *stochastic dependence*. PM2.5 is a
> realistic teaching case because it typically exhibits:
>
> - multiplicative variability (variance scales with level)
> - strong annual seasonality (monthly periodicity)
> - episodic spikes (haze events)
> - nonstationarity in the level series
>
> To keep this notebook self-contained and reproducible, we simulate a PM2.5 series
> with these features.

The simulation is constructed in **log space**:

$$
\log(\text{PM2.5})_t = \text{baseline} + \text{trend}_t + \text{season}_t + \text{spikes}_t + \varepsilon_t
$$

Exponentiating produces strictly positive PM2.5 with multiplicative structure.


In [ ]:
# ============================================================
# PART 2: Simulate PM2.5 + Initial Inspection
# ============================================================

np.random.seed(123)

# --- Simulation parameters ---
n = 240           # 20 years monthly
period = 12
t = np.arange(n)
idx = pd.date_range("2005-01-01", periods=n, freq="MS")

# --- Log-space components ---
baseline = np.log(25)
trend_log = -0.0015 * t
season_log = 0.25 * np.cos(2*np.pi*t/period) - 0.15 * np.sin(2*np.pi*t/period)

# --- Episodic spikes (rare haze events) ---
spike = np.zeros(n)
spike_months = np.random.choice(np.arange(12, n-12), size=10, replace=False)
spike[spike_months] = np.random.uniform(0.4, 0.9, size=len(spike_months))

# --- Noise (log space) ---
eps = np.random.normal(0, 0.22, size=n)

log_pm25 = baseline + trend_log + season_log + spike + eps
pm = pd.Series(np.exp(log_pm25), index=idx, name="PM25")
pm_log = np.log(pm)

# --- Initial inspection: Raw vs Log ---
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(pm.index, pm.values, linewidth=1.2, color="#2c3e50")
axes[0].set_title("PM2.5 (raw) — Multiplicative variance structure", fontsize=12)
axes[0].set_ylabel("PM2.5")
axes[0].grid(True, alpha=0.3)

axes[1].plot(pm_log.index, pm_log.values, linewidth=1.2, color="#2980b9")
axes[1].set_title("log(PM2.5) — Variance stabilized (modeling target)", fontsize=12)
axes[1].set_ylabel("log(PM2.5)")
axes[1].set_xlabel("Date")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("=== SUMMARY STATS (PM2.5, raw) ===")
print(pm.describe().to_string())
print(f"\nSeries length: {n} months ({n//12} years)")
print(f"Seasonality: period = {period} months")
print(f"Spikes: {int(spike.sum() > 0)} months flagged")


## **PART 3 — Identification Using ACF/PACF (With Limits)**

> **Core idea (Shumway & Stoffer, Ch. 3)**
>
> ACF and PACF are **population objects** defined under weak stationarity.
> In Box–Jenkins, they are used to propose *candidate* models — not to uniquely identify them.

---

### **3.1 Reading the ACF/PACF of log(PM2.5)**

Before differencing, the log series may show:

- Slow ACF decay → nonstationarity or strong persistence
- Periodic spikes at lags 12, 24, 36 → seasonality (period = 12)
- PACF behavior guides AR order *only if* the series is stationary

---

### **3.2 Critical Limitation**

> Sample ACF/PACF are **noisy estimators**. Cutoff/decay rules are asymptotic.
> In finite samples:
> - cutoffs are blurred
> - near-unit-root behavior distorts patterns
> - the signal-to-noise ratio matters
>
> **Exam-safe statement:** "ACF/PACF suggest candidate models; they do not uniquely identify the model."


In [ ]:
# ============================================================
# PART 3: ACF/PACF of log(PM2.5)
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

plot_acf(pm_log.dropna(), lags=48, ax=axes[0])
axes[0].set_title("ACF of log(PM2.5)", fontsize=12)

plot_pacf(pm_log.dropna(), lags=48, method="ywm", ax=axes[1])
axes[1].set_title("PACF of log(PM2.5)", fontsize=12)

plt.tight_layout()
plt.show()

print("Observations:")
print("  - ACF decays slowly -> suggests nonstationarity or high persistence")
print("  - Periodic spikes at lags 12, 24, 36 -> seasonality (s=12)")
print("  - PACF has dominant spike at lag 1 + spike at lag 12")
print("  -> Differencing is needed (both regular and possibly seasonal)")


## **PART 4 — Causality vs Invertibility (AR $\neq$ MA)**

> **Why this matters**
>
> Two different conditions are often confused in time series modeling.
> Understanding their distinction is exam-critical.

---

### **4.1 Causality (AR models)**

An AR model is **causal** if current values depend on past shocks in a stable way.

$$
X_t = \sum_{j=0}^{\infty} \psi_j \varepsilon_{t-j} \qquad (\text{causal MA}(\infty) \text{ representation})
$$

This requires all roots of the AR characteristic polynomial $\phi(z) = 1 - \phi_1 z - \cdots - \phi_p z^p$ to lie **outside the unit circle**.

Without causality, AR dynamics can be nonstationary or explosive.

---

### **4.2 Invertibility (MA models)**

An MA model is **invertible** when shocks can be expressed uniquely in terms of past observations:

$$
\varepsilon_t = \sum_{j=0}^{\infty} \pi_j X_{t-j} \qquad (\text{AR}(\infty) \text{ representation})
$$

This requires all roots of the MA characteristic polynomial $\theta(z) = 1 + \theta_1 z + \cdots + \theta_q z^q$ to lie **outside the unit circle**.

Non-invertible MA models can produce the **same autocovariance structure** as invertible ones, making parameters non-unique.

---

### **4.3 The Key Distinction**

| | Causality | Invertibility |
|---|---|---|
| Applies to | AR part | MA part |
| Concerns | How the process is generated | Identifiability and estimation |
| Requires | AR roots outside unit circle | MA roots outside unit circle |
| Failure mode | Nonstationary/explosive | Non-unique parameters |

> **Exam-critical:** Causality concerns generation. Invertibility concerns identifiability. They are not symmetric.


In [ ]:
# ============================================================
# PART 4: Invertibility — theta vs 1/theta produce same ACF
# ============================================================

np.random.seed(42)

def simulate_ma1(theta, n=400):
    eps = np.random.normal(0, 1, n + 1)
    return eps[1:] + theta * eps[:-1]

theta1 = 0.7
theta2 = 1 / theta1  # non-invertible counterpart

x1 = simulate_ma1(theta1, n=2000)
x2 = simulate_ma1(theta2, n=2000)

acf1 = acf(x1, nlags=20, fft=True)
acf2 = acf(x2, nlags=20, fft=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# ACF comparison
axes[0].plot(acf1, linewidth=2.5, marker="o", markersize=4, label=f"$\\theta={theta1:.2f}$ (invertible)")
axes[0].plot(acf2, linewidth=2.5, marker="s", markersize=4, linestyle="--",
             label=f"$\\theta={theta2:.2f}$ (non-invertible)")
axes[0].axhline(0, color="gray", linestyle="-", alpha=0.3)
axes[0].set_title("MA(1) ACF: $\\theta$ vs $1/\\theta$", fontsize=12)
axes[0].set_xlabel("Lag $k$")
axes[0].set_ylabel("ACF")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Theoretical rho(1) for both
thetas = np.linspace(-2, 2, 200)
rho1_vals = thetas / (1 + thetas**2)
axes[1].plot(thetas, rho1_vals, linewidth=2, color="#2c3e50")
axes[1].axvline(theta1, color="#e74c3c", linestyle="--", label=f"$\\theta={theta1}$")
axes[1].axvline(theta2, color="#3498db", linestyle="--", label=f"$\\theta={theta2:.2f}$")
axes[1].axhline(0.5, color="gray", linestyle=":", alpha=0.5)
axes[1].axhline(-0.5, color="gray", linestyle=":", alpha=0.5)
axes[1].set_title("$\\rho(1) = \\theta/(1+\\theta^2)$: Two $\\theta$ values per $\\rho(1)$", fontsize=11)
axes[1].set_xlabel("$\\theta$")
axes[1].set_ylabel("$\\rho(1)$")
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

rho1_a = theta1 / (1 + theta1**2)
rho1_b = theta2 / (1 + theta2**2)
print(f"Theoretical rho(1):")
print(f"  theta = {theta1:.2f}  ->  rho(1) = {rho1_a:.4f}")
print(f"  theta = {theta2:.2f}  ->  rho(1) = {rho1_b:.4f}")
print(f"\n  Same rho(1) -> same autocovariance -> non-identifiable without invertibility.")
print(f"  Convention: always choose |theta| < 1.")


## **PART 5 — Differencing: What It Does to ACF and the Spectrum**

> **Core idea (Shumway & Stoffer)**
>
> Differencing is not a cosmetic step. It **changes the stochastic structure**:
>
> - removes low-frequency components (trend)
> - modifies the ACF
> - often introduces MA-like behavior (overdifferencing creates MA(1))
> - changes the spectrum by suppressing power near frequency 0

---

### **5.1 Regular Differencing**

The first difference operator:

$$
\nabla X_t = X_t - X_{t-1} = (1 - B) X_t
$$

If $X_t$ is a random walk ($X_t = X_{t-1} + \varepsilon_t$), then $\nabla X_t = \varepsilon_t$ (white noise).

---

### **5.2 Seasonal Differencing**

For monthly data with period $s = 12$:

$$
\nabla_{12} X_t = X_t - X_{t-12} = (1 - B^{12}) X_t
$$

Seasonal differencing removes the annual cycle.

---

### **5.3 The ARIMA($p,d,q$) Model**

After $d$ regular differences, the series follows ARMA($p,q$):

$$
\phi(B)\, \nabla^d X_t = \theta(B)\, \varepsilon_t
$$

The full seasonal ARIMA (SARIMA) extends this to:

$$
\phi(B)\,\Phi(B^s)\, \nabla^d \nabla_s^D X_t = \theta(B)\,\Theta(B^s)\, \varepsilon_t
$$

> **Exam-safe statement:** "Differencing trades nonstationarity for dependence. Each difference removes a unit root but may introduce MA structure."


In [ ]:
# ============================================================
# PART 5: Differencing — Time domain + Frequency domain
# ============================================================

# --- Regular differencing ---
d1 = pm_log.diff(1).dropna()

# --- Seasonal differencing (lag 12) ---
d12 = pm_log.diff(12).dropna()

# --- Both: regular + seasonal ---
d1_d12 = pm_log.diff(12).diff(1).dropna()

# --- Time plots ---
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

axes[0].plot(d1.index, d1.values, linewidth=1.0, color="#2c3e50")
axes[0].set_title("$\\nabla \\log(\\mathrm{PM2.5})$ — First difference", fontsize=12)
axes[0].grid(True, alpha=0.3)

axes[1].plot(d12.index, d12.values, linewidth=1.0, color="#8e44ad")
axes[1].set_title("$\\nabla_{12} \\log(\\mathrm{PM2.5})$ — Seasonal difference", fontsize=12)
axes[1].grid(True, alpha=0.3)

axes[2].plot(d1_d12.index, d1_d12.values, linewidth=1.0, color="#27ae60")
axes[2].set_title("$\\nabla \\nabla_{12} \\log(\\mathrm{PM2.5})$ — Both", fontsize=12)
axes[2].set_xlabel("Date")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# --- ACF/PACF of differenced series ---
fig, axes = plt.subplots(2, 2, figsize=(14, 7))

plot_acf(d1, lags=48, ax=axes[0, 0])
axes[0, 0].set_title("ACF: $\\nabla \\log(\\mathrm{PM2.5})$", fontsize=11)

plot_pacf(d1, lags=48, method="ywm", ax=axes[0, 1])
axes[0, 1].set_title("PACF: $\\nabla \\log(\\mathrm{PM2.5})$", fontsize=11)

plot_acf(d1_d12, lags=48, ax=axes[1, 0])
axes[1, 0].set_title("ACF: $\\nabla \\nabla_{12} \\log(\\mathrm{PM2.5})$", fontsize=11)

plot_pacf(d1_d12, lags=48, method="ywm", ax=axes[1, 1])
axes[1, 1].set_title("PACF: $\\nabla \\nabla_{12} \\log(\\mathrm{PM2.5})$", fontsize=11)

plt.tight_layout()
plt.show()

# --- Periodograms ---
periodogram_plot(pm_log.dropna().values, "Periodogram: log(PM2.5)")
periodogram_plot(d1.values, "Periodogram: $\\nabla$ log(PM2.5) — trend removed, seasonality remains")
periodogram_plot(d1_d12.values, "Periodogram: $\\nabla \\nabla_{12}$ log(PM2.5) — both removed")

print("Observations:")
print("  - First diff: removes trend but seasonal spikes remain in ACF at lags 12, 24, 36")
print("  - Seasonal diff alone: removes annual cycle but trend/drift may remain")
print("  - Both: ACF/PACF now show short-range ARMA-like behavior")
print("  - Periodogram confirms: seasonal peaks suppressed after seasonal differencing")


## **PART 6 — Candidate ARIMA Fitting (Done Properly)**

> **What Box–Jenkins actually requires**
>
> Box–Jenkins is iterative:
>
> 1. Identify plausible candidates from ACF/PACF
> 2. Estimate parameters
> 3. Diagnose residuals
> 4. Reject inadequate models
> 5. Iterate (expand candidate set if all fail)

---

### **6.1 First-Pass Candidates (Non-Seasonal)**

After one regular difference of $\log(\text{PM2.5})$, we fit standard ARIMA models:

- ARIMA(1,1,0) — AR(1) after differencing
- ARIMA(0,1,1) — MA(1) after differencing
- ARIMA(1,1,1) — ARMA(1,1) after differencing

These are common first-pass models when one regular difference has been applied.

> **Important:** These models do **not** account for seasonality. We will see what happens when seasonal structure is ignored.


In [ ]:
# ============================================================
# PART 6: Candidate ARIMA Fitting
# ============================================================

candidates = {
    "ARIMA(1,1,0)": (1, 1, 0),
    "ARIMA(0,1,1)": (0, 1, 1),
    "ARIMA(1,1,1)": (1, 1, 1),
}

fits = {}
rows = []

for name, order in candidates.items():
    res = ARIMA(pm_log, order=order).fit()
    fits[name] = res
    rows.append({
        "Model": name,
        "AIC": f"{res.aic:.2f}",
        "BIC": f"{res.bic:.2f}",
        "Log-Lik": f"{res.llf:.2f}"
    })

results_ic = pd.DataFrame(rows)
print("=== Candidate Models (Non-Seasonal) ===")
print(results_ic.to_string(index=False))
print("\nNote: These models ignore seasonality. Diagnostics will reveal this.")


## **PART 7 — Diagnostics: Where Models Live or Die**

> **Exam-critical rule**
>
> A model that fails diagnostics is **incorrect regardless of AIC**.

---

### **7.1 Residual Adequacy Requires**

- **Residual time plot:** no trend, seasonality, or variance patterns
- **Residual ACF:** no significant structure (all within confidence bands)
- **Ljung–Box test:** fail to reject whiteness at multiple lag groups
- **Normality:** approximately Gaussian (for valid prediction intervals)

---

### **7.2 Ljung–Box Test**

The Ljung–Box statistic tests:

$$
H_0: \rho_1 = \rho_2 = \cdots = \rho_h = 0 \quad \text{(residuals are white noise)}
$$

$$
Q(h) = n(n+2) \sum_{k=1}^{h} \frac{\hat{\rho}_k^2}{n-k}
$$

Under $H_0$: $Q(h) \sim \chi^2_{h-p-q}$ (adjusted for estimated parameters).

For monthly data, test at lags 12, 24, 36 to capture seasonal structure.

> **Key principle:** Diagnostics are **necessary conditions** for adequacy. Passing them does not prove the model is correct — but failing them proves it is inadequate.


In [ ]:
# ============================================================
# PART 7: Residual Diagnostics for All Candidates
# ============================================================

print("=" * 60)
print("RESIDUAL DIAGNOSTICS — Non-Seasonal ARIMA Candidates")
print("=" * 60)

for name, res in fits.items():
    print(f"\n{'─' * 60}")
    print(f"  {name}")
    print(f"{'─' * 60}")
    residual_diagnostics(res, name)


## **PART 8 — Why "Lowest AIC" Can Still Be Wrong**

> **Required understanding**

AIC is a **relative criterion**. It ranks models by estimated out-of-sample
Kullback–Leibler risk under the assumption that the candidate class contains
an adequate model.

$$
\mathrm{AIC} = -2 \ln \hat{L} + 2k
$$

where $\hat{L}$ is maximized likelihood and $k$ is the number of parameters.

---

### **8.1 What AIC Does NOT Guarantee**

- Residual independence
- Correct differencing order
- Absence of seasonality
- Robustness to spikes or structural breaks

Therefore, it is possible to observe:

- Model A has lower AIC, but residuals are autocorrelated → **invalid model**
- Model B has higher AIC, but residuals are white → **statistically adequate**

---

### **8.2 The Correct Hierarchy**

$$
\text{Model adequacy} \;\longrightarrow\; \text{Model comparison (AIC/BIC)}
$$

Never the reverse. AIC ranks **adequate models**; it does not create them.

> **Exam-safe statement:** "Model adequacy is a prerequisite for model comparison. AIC ranks adequate models; it does not validate them."


In [ ]:
# ============================================================
# PART 8: AIC vs Whiteness — Demonstration
# ============================================================

print("=" * 60)
print("AIC vs WHITENESS COMPARISON")
print("=" * 60)

alpha = 0.05
summary_rows = []

for name, res in fits.items():
    r = res.resid.dropna()
    lb = acorr_ljungbox(r, lags=[12, 24, 36], return_df=True)
    passes_whiteness = not (lb["lb_pvalue"] < alpha).any()

    summary_rows.append({
        "Model": name,
        "AIC": f"{res.aic:.2f}",
        "Passes Whiteness": "YES" if passes_whiteness else "NO",
        "Min p-value": f"{lb['lb_pvalue'].min():.4f}"
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

passed = [r["Model"] for r in summary_rows if "YES" in r["Passes Whiteness"]]

if len(passed) == 0:
    print("\n*** No non-seasonal candidate passed whiteness. ***")
    print("This means the candidate set is INADEQUATE.")
    print("Seasonal structure must be addressed explicitly.")
    print("-> We need SARIMA (Part 9).")
else:
    print(f"\nModels passing whiteness: {passed}")
    print("If lowest AIC model is not in this set, that demonstrates the AIC lesson.")


## **PART 9 — SARIMA: Handling Seasonality Properly**

> **Why this part matters**
>
> Part 8 showed that all non-seasonal ARIMA candidates failed residual whiteness.
> The ACF of differenced log(PM2.5) has significant spikes at lags 12, 24, 36 —
> this is **seasonal autocorrelation** that non-seasonal models cannot capture.
>
> Box–Jenkins requires us to **iterate**: expand the candidate set to include
> seasonal structure.

---

### **9.1 The SARIMA Model**

SARIMA($p,d,q$)($P,D,Q$)$_s$ combines:

- **Non-seasonal part:** ARIMA($p,d,q$) for short-range dependence
- **Seasonal part:** ARIMA($P,D,Q$)$_s$ for periodic dependence at lag $s$

The full model:

$$
\phi(B)\,\Phi(B^s)\;\nabla^d \nabla_s^D\, X_t = \theta(B)\,\Theta(B^s)\;\varepsilon_t
$$

where:
- $\phi(B)$: non-seasonal AR polynomial of order $p$
- $\Phi(B^s)$: seasonal AR polynomial of order $P$
- $\theta(B)$: non-seasonal MA polynomial of order $q$
- $\Theta(B^s)$: seasonal MA polynomial of order $Q$
- $\nabla^d = (1-B)^d$: regular differencing
- $\nabla_s^D = (1-B^s)^D$: seasonal differencing

---

### **9.2 Identification Strategy**

After applying $\nabla \nabla_{12}$ (regular + seasonal differencing):

1. Look at **lags 1–3** in ACF/PACF → non-seasonal orders ($p, q$)
2. Look at **lags 12, 24, 36** in ACF/PACF → seasonal orders ($P, Q$)
3. ACF cutoff at lag 12 → seasonal MA ($Q = 1$)
4. PACF cutoff at lag 12 → seasonal AR ($P = 1$)

---

### **9.3 Common SARIMA for Monthly Data**

For monthly environmental data, the following are typical first candidates:

- SARIMA(1,1,1)(0,1,1)$_{12}$ — "airline model" variant
- SARIMA(0,1,1)(0,1,1)$_{12}$ — pure MA structure
- SARIMA(1,1,0)(1,1,0)$_{12}$ — pure AR structure


In [ ]:
# ============================================================
# PART 9: SARIMA Fitting + Diagnostics
# ============================================================

from statsmodels.tsa.statespace.sarimax import SARIMAX

sarima_candidates = {
    "SARIMA(1,1,1)(0,1,1)12": {"order": (1,1,1), "seasonal_order": (0,1,1,12)},
    "SARIMA(0,1,1)(0,1,1)12": {"order": (0,1,1), "seasonal_order": (0,1,1,12)},
    "SARIMA(1,1,0)(1,1,0)12": {"order": (1,1,0), "seasonal_order": (1,1,0,12)},
    "SARIMA(1,1,1)(1,1,0)12": {"order": (1,1,1), "seasonal_order": (1,1,0,12)},
}

sarima_fits = {}
sarima_rows = []

for name, spec in sarima_candidates.items():
    try:
        model = SARIMAX(pm_log, order=spec["order"],
                        seasonal_order=spec["seasonal_order"],
                        enforce_stationarity=True,
                        enforce_invertibility=True)
        res = model.fit(disp=False)
        sarima_fits[name] = res
        sarima_rows.append({
            "Model": name,
            "AIC": f"{res.aic:.2f}",
            "BIC": f"{res.bic:.2f}",
            "Log-Lik": f"{res.llf:.2f}"
        })
    except Exception as e:
        print(f"  {name}: FAILED ({e})")

sarima_ic = pd.DataFrame(sarima_rows)
print("=== SARIMA Candidates ===")
print(sarima_ic.to_string(index=False))

# --- Diagnostics for each SARIMA ---
print("\n" + "=" * 60)
print("SARIMA RESIDUAL DIAGNOSTICS")
print("=" * 60)

alpha = 0.05
sarima_summary = []

for name, res in sarima_fits.items():
    print(f"\n{'─' * 60}")
    print(f"  {name}")
    print(f"{'─' * 60}")
    residual_diagnostics(res, name)

    r = res.resid.dropna()
    lb = acorr_ljungbox(r, lags=[12, 24, 36], return_df=True)
    passes = not (lb["lb_pvalue"] < alpha).any()
    sarima_summary.append({"Model": name, "AIC": f"{res.aic:.2f}", "Passes": "YES" if passes else "NO"})

print("\n" + "=" * 60)
print("SARIMA SUMMARY")
print("=" * 60)
print(pd.DataFrame(sarima_summary).to_string(index=False))

passed_sarima = [r["Model"] for r in sarima_summary if r["Passes"] == "YES"]
if passed_sarima:
    print(f"\nAdequate models: {passed_sarima}")
    print("These can now be ranked by AIC/BIC for final selection.")
else:
    print("\nNo SARIMA candidate passed. Further iteration needed.")


## **PART 10 — Final Model Defense (Exam Template)**

> **How you defend your final model in a comprehensive exam**

---

### **What you do NOT say:**

> "I used auto_arima and got ARIMA($p,d,q$)."

---

### **What you say:**

1. I **transformed** the series to stabilize variance (log scale).
2. I inspected ACF/PACF and the **periodogram** to identify nonstationarity and seasonality.
3. I applied **regular and seasonal differencing**, justified by ACF behavior and ADF testing.
4. I proposed **multiple candidate ARIMA models** consistent with ACF/PACF patterns.
5. I showed that **non-seasonal models failed** residual diagnostics (Ljung–Box rejection at seasonal lags).
6. I expanded to **SARIMA candidates** to address seasonal structure.
7. I compared adequate models using **AIC/BIC**, selecting the most parsimonious adequate model.
8. I verified **residual whiteness** (Ljung–Box), approximate **normality** (histogram), and **parameter stability**.

---

### **Key Principles**

- Model adequacy is a **prerequisite** for comparison — AIC ranks adequate models only.
- Diagnostics are **necessary conditions** — passing does not prove correctness, but failing proves inadequacy.
- The Box–Jenkins pipeline is **iterative** — if all candidates fail, expand the candidate set.
- Parsimony matters — among adequate models, prefer simpler ones.

> **Exam-safe closing:** "A good ARIMA model is not the one with the lowest AIC. A good ARIMA model is the one with residuals closest to white noise, stable parameters, and justifiable structure."


## **PART 11 — Self-Test: Exam Preparation Questions**

> Work through these without looking at notes first.

---

### **Conceptual Questions (Oral Exam Style)**

**Q1.** What are the steps of the Box–Jenkins pipeline? Why must the series be fixed before modeling begins?

**Q2.** You observe strong spikes at lags 12, 24, 36 in the residual ACF of an ARIMA(1,1,1) model fitted to monthly data. What does this indicate? What should you do?

**Q3.** Explain the difference between causality and invertibility. Which applies to AR models? Which to MA?

**Q4.** What does differencing do to the ACF? What does it do to the spectrum? Can you overdifference?

**Q5.** Your colleague selects a model based on lowest AIC without checking residual diagnostics. Explain why this is incorrect and give a concrete scenario where it fails.

**Q6.** Write the full SARIMA($p,d,q$)($P,D,Q$)$_s$ model in backshift notation. Explain each component.

**Q7.** Why is the Ljung–Box test applied at multiple lag groups (12, 24, 36) for monthly data? What would testing at only lag 5 miss?

---

### **Technical Questions**

**Q8.** For SARIMA(1,1,1)(0,1,1)$_{12}$:
- How many parameters are estimated?
- What is the total order of differencing?
- Write the model in backshift notation.

**Q9.** You fit two models to the same series:
- Model A: AIC = -120, Ljung–Box p = 0.002 at lag 12
- Model B: AIC = -115, Ljung–Box p = 0.45 at lag 12
- Which do you select and why?

**Q10.** Explain why applying both $\nabla$ and $\nabla_{12}$ to a series with trend and seasonality is different from applying $\nabla_{13}$. What does each remove?

---

### **Practical Challenge**

**Q11.** Modify the code in Part 6 to add ARIMA(2,1,0) and ARIMA(0,1,2) to the candidate set. Do either pass diagnostics? Why or why not?

**Q12.** In Part 9, try SARIMA(0,1,0)(0,1,1)$_{12}$ — the simplest seasonal-only model. Does it pass? What does this tell you about the relative importance of seasonal vs non-seasonal structure in this data?
